# SWAP-Stress: CONUS Inference Pipeline

This notebook walks through the full CONUS-scale inference pipeline:

1. SMAP L3 daily coverage and gap-fill
2. Static feature stack vs model features
3. Running predictions (guarded)
4. Gap-fill coverage: before vs after
5. Spatial sanity check (July 2024 mean/std)
6. Temporal sanity at 5 representative locations


In [ ]:
from __future__ import annotations

import json
import os
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import rasterio
except Exception:
    rasterio = None

try:
    import geopandas as gpd
except Exception:
    gpd = None

# ------------------------------------------------------------------
# Config
# ------------------------------------------------------------------
DATA_ROOT = os.environ.get("SWAPSTRESS_DATA_ROOT", "/nas/soils")
DATA_ROOT = str(Path(DATA_ROOT).expanduser())

PRED_DIR = os.path.join(
    DATA_ROOT, "swapstress", "inference", "predictions", "direct_rf_9km_global_pruned"
)
GF_DIR = PRED_DIR + "_gapfilled"
SMAP_DIR = os.path.join(DATA_ROOT, "smap", "SPL3SMP_E", "daily_tif")
SMAP_GF = SMAP_DIR + "_gapfilled"
FEAT_DIR = os.path.join(DATA_ROOT, "swapstress", "inference", "conus_features")
MODEL_DIR = os.path.join(
    DATA_ROOT, "swapstress", "models", "direct_rf_9km_global_pruned"
)

US_STATES_SHP = "/nas/boundaries/us_states_tiger_wgs.shp"
OUT_DIR = Path("notebooks/_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Guard flags — set True to re-run expensive operations
RUN_PREDICT = False

# Load US state boundaries once
states = None
if gpd is not None and os.path.exists(US_STATES_SHP):
    states = gpd.read_file(US_STATES_SHP).to_crs(epsg=6933)

print("DATA_ROOT:", DATA_ROOT)
print("MODEL_DIR:", MODEL_DIR)
print("PRED_DIR:", PRED_DIR)
print("GF_DIR:", GF_DIR)

## 2) SMAP L3 Daily Coverage

Each SMAP L3 pass covers roughly 31% of CONUS pixels per day — the remainder
are gaps filled by the gap-fill step.  Here we count valid pixels per file and
show one raw day alongside its gap-filled counterpart.


In [ ]:
from map.inference.gapfill_conus import discover_source_rasters

assert rasterio is not None, "rasterio is required"


# Discover all SMAP daily TIFs
smap_files = discover_source_rasters(SMAP_DIR, prefix="smap_sm")
smap_gf_files = discover_source_rasters(SMAP_GF, prefix="smap_sm")

print(f"Raw SMAP files:       {len(smap_files):,}")
print(f"Gap-filled SMAP:      {len(smap_gf_files):,}")

# Count valid pixels per raw day
smap_dates = sorted(smap_files)
valid_counts = []
for d in smap_dates:
    with rasterio.open(smap_files[d]) as src:
        arr = src.read(1)
        valid_counts.append(int(np.sum(~np.isnan(arr))))

valid_arr = np.array(valid_counts)
total_px = arr.size  # same shape for all days
print(
    f"\nMedian valid px/day: {np.median(valid_arr):,.0f} / {total_px:,} "
    f"({100 * np.median(valid_arr) / total_px:.1f}%)"
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(
    [d.toordinal() for d in smap_dates], valid_arr, lw=0.6, color="steelblue", alpha=0.8
)
ax.axhline(
    np.median(valid_arr),
    color="tomato",
    lw=1.2,
    linestyle="--",
    label=f"Median {np.median(valid_arr):,.0f} px",
)
ax.set_xlabel("Date")
ax.set_ylabel("Valid pixels")
ax.set_title("SMAP L3 valid-pixel count per day (raw)")
ax.legend()
# Re-label x axis with real dates
ticks = [d for d in smap_dates if d.month == 1 and d.day == 1]
ax.set_xticks([d.toordinal() for d in ticks])
ax.set_xticklabels([str(d.year) for d in ticks], rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
# Side-by-side: one raw SMAP day vs gap-filled counterpart
sample_date = date(2024, 6, 15)
assert sample_date in smap_files, f"{sample_date} not in raw SMAP files"
assert sample_date in smap_gf_files, f"{sample_date} not in gap-filled SMAP files"

with rasterio.open(smap_files[sample_date]) as src:
    raw_arr = src.read(1)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

with rasterio.open(smap_gf_files[sample_date]) as src:
    gf_arr = src.read(1)

valid_raw = raw_arr[~np.isnan(raw_arr)]
vmin, vmax = np.percentile(valid_raw, [2, 98])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for ax, arr, title in [
    (ax1, raw_arr, f"Raw SMAP — {sample_date} ({np.sum(~np.isnan(arr)):,} valid px)"),
    (ax2, gf_arr, f"Gap-filled SMAP — {sample_date}"),
]:
    im = ax.imshow(
        arr, extent=extent, origin="upper", cmap="viridis", vmin=vmin, vmax=vmax
    )
    if states is not None:
        states.boundary.plot(ax=ax, color="0.3", linewidth=0.3)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02, label="VWC (m³/m³)")
    ax.set_title(title, fontsize=11)
    ax.tick_params(labelsize=7)

fig.tight_layout()
plt.show()

## 3) Static Feature Stack vs Model Features

The model was trained on a specific ordered feature list saved in
`direct_rf_features.json`.  Here we cross-check which features map to which
file / band in the CONUS feature rasters, then plot four representative bands.


In [ ]:
# Load model feature list
feat_json = os.path.join(MODEL_DIR, "direct_rf_features.json")
with open(feat_json) as f:
    model_features = json.load(f)
print(f"Model expects {len(model_features)} features")
print("First 10:", model_features[:10])

# Scan all *_ease2.tif files and build (file, band_index, band_name) table
band_inventory = []
for fname in sorted(os.listdir(FEAT_DIR)):
    if not fname.endswith("_ease2.tif"):
        continue
    fpath = os.path.join(FEAT_DIR, fname)
    with rasterio.open(fpath) as src:
        for i in range(src.count):
            band_inventory.append(
                {
                    "file": fname,
                    "band_index": i + 1,
                    "band_name": src.descriptions[i] or f"band_{i + 1}",
                }
            )

inv_df = pd.DataFrame(band_inventory)
print(f"\nTotal bands in FEAT_DIR: {len(inv_df)}")

# Cross-check: which model features are found in the raster inventory?
static_features = [
    f for f in model_features if f not in ("theta", "depth_cm", "rosetta_level")
]
matched = inv_df[inv_df["band_name"].isin(static_features)]
missing_in_rasters = set(static_features) - set(matched["band_name"])

print(f"\nModel static features: {len(static_features)}")
print(f"Found in rasters:      {len(matched)}")
if missing_in_rasters:
    print(f"Missing from rasters:  {sorted(missing_in_rasters)}")
else:
    print("All static features found in rasters.")

matched.head(20)

In [ ]:
def plot_bands(tif_path, band_names, ncols=2, cmap="viridis", title=None):
    """Plot selected bands from a multi-band GeoTIFF as a grid of maps."""
    with rasterio.open(tif_path) as src:
        descs = [src.descriptions[i] or f"band_{i + 1}" for i in range(src.count)]
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
        indices = []
        for name in band_names:
            if name in descs:
                indices.append(descs.index(name))
            else:
                print(f"  Warning: '{name}' not found in {os.path.basename(tif_path)}")
        if not indices:
            return None
        bands = [src.read(i + 1) for i in indices]

    n = len(indices)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).flat

    for i, (arr, idx) in enumerate(zip(bands, indices)):
        ax = axes[i]
        valid = arr[~np.isnan(arr)]
        vmin, vmax = np.percentile(valid, [2, 98]) if len(valid) else (0, 1)
        im = ax.imshow(
            arr, extent=extent, origin="upper", cmap=cmap, vmin=vmin, vmax=vmax
        )
        if states is not None:
            states.boundary.plot(ax=ax, color="0.3", linewidth=0.3)
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
        ax.set_title(descs[idx], fontsize=10)
        ax.tick_params(labelsize=7)

    for j in range(n, nrows * ncols):
        axes[j].set_visible(False)

    if title:
        fig.suptitle(title, fontsize=13, y=1.01)
    fig.tight_layout()
    return fig


# Plot 4 representative bands used by the model
fig = plot_bands(
    os.path.join(FEAT_DIR, "soilgrids_ease2.tif"),
    ["clay_0-5cm_mean", "bdod_0-5cm_mean"],
    ncols=2,
    cmap="YlOrBr",
    title="SoilGrids (representative model features)",
)
plt.show()

fig = plot_bands(
    os.path.join(FEAT_DIR, "terrain_ease2.tif"),
    ["elevation"],
    ncols=1,
    cmap="terrain",
    title="Terrain elevation",
)
plt.show()

fig = plot_bands(
    os.path.join(FEAT_DIR, "smap_l3_clim_ease2.tif"),
    ["vegetation_water_content_am_mean"],
    ncols=1,
    cmap="YlGn",
    title="SMAP L3 VWC climatology",
)
plt.show()

fig = plot_bands(
    os.path.join(FEAT_DIR, "polaris_ease2.tif"),
    ["ksat_mean"],
    ncols=1,
    cmap="viridis",
    title="POLARIS Ksat",
)
plt.show()

## 4) Running Inference

`RUN_PREDICT = False` by default — set to `True` to run a single-day demo.

### CLI invocation

```bash
uv run python -m map.inference.predict_conus \
    --model-dir /nas/soils/swapstress/models/direct_rf_9km_global_pruned \
    --static-dir /nas/soils/swapstress/inference/conus_features \
    --smap-dir   /nas/soils/smap/SPL3SMP_E/daily_tif \
    --output-dir /nas/soils/swapstress/inference/predictions/direct_rf_9km_global_pruned \
    --start-date 20240101 \
    --end-date   20241231 \
    --depth-cm   5 \
    --batch-size 50000
```

**Flags:**
- `--model-dir`: directory containing `direct_rf_model.joblib`, `direct_rf_imputer.joblib`, `direct_rf_features.json`
- `--static-dir`: directory of `*_ease2.tif` static covariate rasters
- `--smap-dir`: directory of daily `smap_sm_YYYYMMDD.tif` rasters
- `--output-dir`: where to write `suction_YYYYMMDD.tif` outputs
- `--depth-cm`: soil depth for prediction (controls `rosetta_level`)
- `--batch-size`: pixels per RF prediction batch (memory vs speed trade-off)


In [ ]:
if RUN_PREDICT:
    from map.inference.predict_conus import run_prediction

    demo_date = "20240615"
    run_prediction(
        model_dir=MODEL_DIR,
        static_dir=FEAT_DIR,
        smap_dir=SMAP_DIR,
        output_dir=PRED_DIR,
        start_date=demo_date,
        end_date=demo_date,
        depth_cm=5.0,
        rosetta_level=None,
        batch_size=50_000,
        overwrite=False,
        write_linear=False,
    )

    # Show output raster
    out_tif = os.path.join(PRED_DIR, f"suction_{demo_date}.tif")
    with rasterio.open(out_tif) as src:
        arr = src.read(1)
        ex = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    valid = arr[arr != -9999]
    print(f"Valid pixels: {len(valid):,} / {arr.size:,}")

    fig, ax = plt.subplots(figsize=(12, 6))
    vmin, vmax = np.percentile(valid, [2, 98])
    im = ax.imshow(arr, extent=ex, origin="upper", cmap="RdBu", vmin=vmin, vmax=vmax)
    if states is not None:
        states.boundary.plot(ax=ax, color="0.3", linewidth=0.3)
    ax.set_xlim(ex[0], ex[1])
    ax.set_ylim(ex[2], ex[3])
    fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02, label="log10 suction (cm)")
    ax.set_title(f"Predicted suction — {demo_date}")
    fig.tight_layout()
    plt.show()
else:
    print("RUN_PREDICT=False  — skipping inference demo")

## 5) Gap-fill: Before vs After

Raw suction rasters follow SMAP coverage: ~52k median valid pixels per day.
Gap-filled rasters cover all 121,435 CONUS pixels every day.


In [ ]:
# Discover raw and gap-filled suction rasters
raw_suction = discover_source_rasters(PRED_DIR, prefix="suction")
gf_suction = discover_source_rasters(GF_DIR, prefix="suction")

print(f"Raw suction rasters:        {len(raw_suction):,}")
print(f"Gap-filled suction rasters: {len(gf_suction):,}")


# Count valid pixels per day
def count_valid(raster_dict, nodata=-9999.0):
    counts = {}
    for d, p in sorted(raster_dict.items()):
        with rasterio.open(p) as src:
            arr = src.read(1).astype(np.float32)
        counts[d] = int(np.sum(arr != nodata))
    return counts


raw_counts = count_valid(raw_suction)
gf_counts = count_valid(gf_suction)

raw_dates = sorted(raw_counts)
gf_dates = sorted(gf_counts)

print(f"\nRaw  — median valid px/day: {np.median(list(raw_counts.values())):,.0f}")
print(f"GF   — median valid px/day: {np.median(list(gf_counts.values())):,.0f}")

In [ ]:
# Bar chart: valid pixels per day, raw vs gap-filled
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(
    [d.toordinal() for d in raw_dates],
    [raw_counts[d] for d in raw_dates],
    width=1,
    color="steelblue",
    alpha=0.7,
    label="Raw",
)
ax.bar(
    [d.toordinal() for d in gf_dates],
    [gf_counts[d] for d in gf_dates],
    width=1,
    color="tomato",
    alpha=0.4,
    label="Gap-filled",
)

ax.set_xlabel("Date")
ax.set_ylabel("Valid pixels")
ax.set_title("Suction raster valid-pixel count: raw vs gap-filled")
ax.legend()

all_dates_ord = [d.toordinal() for d in sorted(set(raw_dates) | set(gf_dates))]
year_ticks = [
    d for d in sorted(set(raw_dates) | set(gf_dates)) if d.month == 1 and d.day == 1
]
ax.set_xticks([d.toordinal() for d in year_ticks])
ax.set_xticklabels([str(d.year) for d in year_ticks], rotation=45)
fig.tight_layout()

out = OUT_DIR / "07_gapfill_coverage.png"
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", out)

In [ ]:
# Side-by-side map: same date raw vs gap-filled suction
show_date = date(2024, 6, 15)
assert show_date in raw_suction, f"{show_date} not found in raw suction"
assert show_date in gf_suction, f"{show_date} not found in gf suction"

with rasterio.open(raw_suction[show_date]) as src:
    raw_arr = src.read(1).astype(np.float32)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

with rasterio.open(gf_suction[show_date]) as src:
    gf_arr = src.read(1).astype(np.float32)

raw_arr[raw_arr == -9999] = np.nan
gf_arr[gf_arr == -9999] = np.nan

all_valid = np.concatenate([raw_arr[~np.isnan(raw_arr)], gf_arr[~np.isnan(gf_arr)]])
vmin, vmax = np.percentile(all_valid, [2, 98])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for ax, arr, label in [
    (ax1, raw_arr, f"Raw — {show_date} ({np.sum(~np.isnan(raw_arr)):,} px)"),
    (ax2, gf_arr, f"Gap-filled — {show_date} ({np.sum(~np.isnan(gf_arr)):,} px)"),
]:
    im = ax.imshow(
        arr, extent=extent, origin="upper", cmap="RdBu", vmin=vmin, vmax=vmax
    )
    if states is not None:
        states.boundary.plot(ax=ax, color="0.3", linewidth=0.3)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02, label="log10 suction (cm)")
    ax.set_title(label, fontsize=11)
    ax.tick_params(labelsize=7)

fig.tight_layout()
plt.show()

## 6) Spatial Sanity — July 2024

Load all gap-filled suction rasters for July 2024, compute pixel-wise mean and
standard deviation.  Expected pattern: high suction (dry) in the arid West;
lower suction (wetter) in the humid East; Rocky Mountain signatures visible.


In [ ]:
# Load July 2024 gap-filled rasters
july_dates = [d for d in sorted(gf_suction) if d.year == 2024 and d.month == 7]
print(f"July 2024 gap-filled rasters: {len(july_dates)}")

assert len(july_dates) > 0, "No July 2024 gap-filled suction rasters found"

with rasterio.open(gf_suction[july_dates[0]]) as src:
    shape = (src.height, src.width)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

stack = np.full((len(july_dates), *shape), np.nan, dtype=np.float32)
for i, d in enumerate(july_dates):
    with rasterio.open(gf_suction[d]) as src:
        arr = src.read(1).astype(np.float32)
        arr[arr == -9999] = np.nan
        stack[i] = arr

mean_suction = np.nanmean(stack, axis=0)
std_suction = np.nanstd(stack, axis=0)

print(
    f"Mean suction range: {np.nanmin(mean_suction):.2f} – "
    f"{np.nanmax(mean_suction):.2f} log10(cm)"
)
print(
    f"Std  suction range: {np.nanmin(std_suction):.3f} – "
    f"{np.nanmax(std_suction):.3f} log10(cm)"
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for ax, arr, title, cmap in [
    (ax1, mean_suction, "Mean log10 suction — July 2024", "RdBu_r"),
    (ax2, std_suction, "Std dev log10 suction — July 2024", "RdBu_r"),
]:
    valid = arr[~np.isnan(arr)]
    vmin, vmax = np.percentile(valid, [2, 98])
    im = ax.imshow(arr, extent=extent, origin="upper", cmap=cmap, vmin=vmin, vmax=vmax)
    if states is not None:
        states.boundary.plot(ax=ax, color="0.3", linewidth=0.3)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
    ax.set_title(title, fontsize=12)
    ax.tick_params(labelsize=7)

fig.tight_layout()
out = OUT_DIR / "07_spatial_sanity.png"
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", out)

## 7) Temporal Sanity — 5 Representative Pixels

Sample daily gap-filled suction and raw SMAP VWC at five CONUS locations across
2024.  We expect an anti-correlation (high VWC → low suction) and a seasonal
cycle (wetter winters/springs, drier summers in most locations).


In [ ]:
# Five representative locations (lat, lon, label)
SITES = [
    (47.5, -122.0, "Pacific NW"),
    (41.0, -100.0, "Great Plains"),
    (33.0, -87.0, "SE"),
    (33.5, -111.5, "SW desert"),
    (42.0, -93.0, "Midwest"),
]


def latlon_to_rowcol(lat, lon, transform, crs_from="EPSG:4326", crs_to=None):
    """Convert geographic lat/lon to raster row/col."""
    from pyproj import Transformer

    if crs_to is None:
        crs_to = "EPSG:6933"
    transformer = Transformer.from_crs(crs_from, crs_to, always_xy=True)
    x, y = transformer.transform(lon, lat)
    col = int((x - transform.c) / transform.a)
    row = int((y - transform.f) / transform.e)
    return row, col


# Get transform from a reference raster
ref_date = sorted(gf_suction)[0]
with rasterio.open(gf_suction[ref_date]) as src:
    transform = src.transform
    height, width = src.height, src.width

site_rowcols = []
for lat, lon, label in SITES:
    row, col = latlon_to_rowcol(lat, lon, transform)
    row = max(0, min(row, height - 1))
    col = max(0, min(col, width - 1))
    site_rowcols.append((row, col, label))
    print(f"{label:15s} lat={lat:+.1f} lon={lon:+.1f} → row={row} col={col}")

In [ ]:
# Extract time series for all 2024 gap-filled suction rasters
year2024_dates = [d for d in sorted(gf_suction) if d.year == 2024]
print(f"2024 gap-filled suction days: {len(year2024_dates)}")

# Build (n_days, n_sites) arrays for suction and SMAP VWC
suction_ts = np.full((len(year2024_dates), len(SITES)), np.nan)
vwc_ts = np.full((len(year2024_dates), len(SITES)), np.nan)

for i, d in enumerate(year2024_dates):
    with rasterio.open(gf_suction[d]) as src:
        arr = src.read(1).astype(np.float32)
        arr[arr == -9999] = np.nan
    for j, (row, col, _) in enumerate(site_rowcols):
        suction_ts[i, j] = arr[row, col]

    if d in smap_files:
        with rasterio.open(smap_files[d]) as src:
            arr_sm = src.read(1)
        for j, (row, col, _) in enumerate(site_rowcols):
            vwc_ts[i, j] = arr_sm[row, col]

print("Extracted suction and VWC time series for all sites.")

In [ ]:
import matplotlib.dates as mdates

fig, axes = plt.subplots(len(SITES), 1, figsize=(14, 3 * len(SITES)), sharex=True)

date_objs = [datetime(d.year, d.month, d.day) for d in year2024_dates]

for ax, (row, col, label), j in zip(axes, site_rowcols, range(len(SITES))):
    suc = suction_ts[:, j]
    vwc = vwc_ts[:, j]

    ax2 = ax.twinx()
    ax2.plot(date_objs, vwc, color="steelblue", lw=0.8, alpha=0.7, label="VWC")
    ax2.set_ylabel("VWC (m³/m³)", color="steelblue", fontsize=8)
    ax2.tick_params(axis="y", labelcolor="steelblue", labelsize=7)
    ax2.invert_yaxis()  # high VWC → low on axis (anti-correlation visual)

    ax.plot(date_objs, suc, color="tomato", lw=1.0, label="Suction")
    ax.set_ylabel("log10 suction (cm)", color="tomato", fontsize=8)
    ax.tick_params(axis="y", labelcolor="tomato", labelsize=7)
    ax.set_title(
        f"{label} — lat={SITES[j][0]:+.1f}, lon={SITES[j][1]:+.1f}", fontsize=10
    )

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b"))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].set_xlabel("2024")

fig.suptitle("Daily suction (red) vs SMAP VWC (blue, inverted) — 2024", fontsize=13)
fig.tight_layout()
out = OUT_DIR / "07_temporal_sanity.png"
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", out)